In [ ]:
!pip install -U langchain
!pip install -U langchain-cohere
!pip install -U langchain-community
!pip install -U langchain-text-splitters
!pip install -U langchain-chroma
!pip install -U pypdf
!pip install -U chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.0/148.0 kB 15.1 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.17
    Uninstalling langchain-1.3.17:
      Successfully uninstalled langchain-1.3.17
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not c

In [ ]:
!pip install -U langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 21.7 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.0
    Uninstalling langchain-core-1.6.0:
      Successfully uninstalled langchain-core-1.6.0


In [3]:
from google.colab import userdata
key=userdata.get('coherkey')

# **Part 1 — Test the RAG with Your Own CV**

## **Load Pdf**

In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/Ahmed-Maged-Motea-FlowCV-Resume-20251008.pdf")

documents = loader.load()

print("Number of pages:", len(documents))

print(documents[0].page_content[:500])

/tmp/ipykernel_2695/1831675838.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Number of pages: 2
Ahmed Maged Motea
Al & ML Engineer
ahmedmaged2004zsc@gmail.com
 
01026381342
 
in/ahmed-maged-swe
 
github.com/Ahmd00z
 
SUMMARY
AI & ML Engineer with a solid foundation in computer engineering and practical experience in 
designing and implementing machine learning models. Proficient in deep learning, computer vision, 
and natural language processing, with a strong enthusiasm for creating scalable systems and 
leveraging AI to address real-world challenges.
EDUCATION
Faculty Of Engineering, Zag


In [5]:
from langchain_cohere import CohereEmbeddings

embeddings = CohereEmbeddings(
    model="embed-v4.0",
    cohere_api_key=key
)

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=80
)

chunks = text_splitter.split_documents(documents)

In [7]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="pdf_rag"
)

In [8]:
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 5
    }
)

In [9]:
query = "What is this document about?"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):

    print(f"\n--- Document {i+1} ---")

    print(doc.page_content)

    print("Metadata:", doc.metadata)


--- Document 1 ---
Ahmed Maged Motea
Al & ML Engineer
ahmedmaged2004zsc@gmail.com
 
01026381342
 
in/ahmed-maged-swe
 
github.com/Ahmd00z
 
SUMMARY
AI & ML Engineer with a solid foundation in computer engineering and practical experience in 
designing and implementing machine learning models. Proficient in deep learning, computer vision, 
and natural language processing, with a strong enthusiasm for creating scalable systems and 
leveraging AI to address real-world challenges.
EDUCATION
Faculty Of Engineering, Zagazig University
B.Sc. Computer Engineering
Metadata: {'creator': 'FlowCV - https://flowcv.com', 'page_label': '1', 'source': '/content/Ahmed-Maged-Motea-FlowCV-Resume-20251008.pdf', 'page': 0, 'keywords': 'FlowCV – Online Resume Builder – https://flowcv.com', 'creationdate': '2025-10-07T21:18:29+00:00', 'moddate': '2025-10-07T21:18:29+00:00', 'total_pages': 2, 'producer': 'Skia/PDF m141'}

--- Document 2 ---
cases.Implemented a Voting Classifier and Neural Network to address 

## Generation


In [10]:
from langchain_cohere import ChatCohere

llm = ChatCohere(
    model="command-a-03-2025",
    temperature=0,
    cohere_api_key=key
)


## Prompt

In [11]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer the user's question using ONLY the context below.

If the answer cannot be found in the context,
say "I don't know."

Context:
{context}## top 1

Question:
{question}
""")

## RAG Chain

In [12]:
from langchain_core.runnables import RunnablePassthrough


In [13]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )


rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

In [14]:
question = "What is the main topic of the document?"

response = rag_chain.invoke(question)

print(response.content)

The main topic of the document is the professional profile and experience of **Ahmed Maged Motea**, an **AI & ML Engineer**. It highlights his education, skills, projects, and work experience in the field of artificial intelligence and machine learning.


In [15]:
question = "What is  ahmed experience ?"

response = rag_chain.invoke(question)

print(response.content)

Ahmed Maged Motea has experience as an **AI & Data Science Engineer** at **DEPI** from **06/2025 – Present**. In this role, he:

- Developed and deployed **ML/DL models** for predictive analytics and automation.  
- Performed **data preprocessing**, **feature engineering**, and **optimization** to enhance model accuracy.  
- Applied **AI techniques** (Computer Vision, Natural Language Processing, predictive modeling) to deliver insights and scalable solutions.  

Additionally, he has worked on projects such as:  
1. **AI-Powered Predictive Maintenance**: Developed a system using IoT sensor data and advanced ML models to forecast equipment failures, implemented pipelines to optimize maintenance, and deployed the solution using scalable MLOps practices.  
2. **Credit Card Fraud Detection System**: Handled a highly unbalanced Kaggle dataset of 28,807 transactions with only 92 fraud cases.  

His experience also includes relevant coursework in **Machine Learning**, **Artificial Intelligenc

# pipline

In [19]:
def AI (documents,chsz,chov):
  # from langchain_google_genai import GoogleGenerativeAIEmbeddings # This line was causing the error

  from langchain_cohere import CohereEmbeddings

  embeddings = CohereEmbeddings(
      model="embed-v4.0",
      cohere_api_key=key
  )
  from langchain_text_splitters import RecursiveCharacterTextSplitter

  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=chsz,
      chunk_overlap=chov
  )

  chunks = text_splitter.split_documents(documents)
  from langchain_chroma import Chroma

  vectorstore = Chroma.from_documents(
      documents=chunks,
      embedding=embeddings,
      collection_name="pdf_rag"
  )

  retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 5
    }
  )

  from langchain_cohere import ChatCohere

  llm = ChatCohere(
      model="command-a-03-2025",
      temperature=0,
      cohere_api_key=key
  )


  from langchain_core.prompts import ChatPromptTemplate

  prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer the user's question using ONLY the context below.

If the answer cannot be found in the context,
say "I don't know."

Context:
{context}## top 1

Question:
{question}
""")

  from langchain_core.runnables import RunnablePassthrough
  def format_docs(docs):
      return "\n\n".join(
          doc.page_content
          for doc in docs
      )


  rag_chain = (
      {
          "context": retriever | format_docs,
          "question": RunnablePassthrough()
      }
      | prompt
      | llm
  )
  return rag_chain , retriever

In [20]:
rag_chain,retriever=AI(documents,600,100)

In [21]:
question = "What are my main technical skills?"

response = rag_chain.invoke(question)

print(response.content)

Your main technical skills, based on the provided context, include:

1. **Programming Languages**: C, C++, Java, Python  
2. **Tools**: Git & Github, Visual Studio Code, Active VHDL  
3. **Software Libraries**: Numpy, Pandas, Matplotlib, Seaborn, Scikit-learn, OpenCV, TensorFlow, PyTorch  
4. **Deep Learning & Machine Learning**: CNNs, RNNs, GNNs, GAT, VAE, LSTMs, Transformers, Supervised Learning, Unsupervised Learning  
5. **Software Basics**: Problem solving, Object-Oriented Programming (OOP), Data Structures & Algorithms (DSA), Databases  

These skills are highlighted in the "SKILLS" section of the context.


In [22]:
question = "What machine learning experience do I have?"

response = rag_chain.invoke(question)

print(response.content)

Based on the context provided, your machine learning experience includes:

1. **Developing and deploying ML/DL models** for predictive analytics and automation at DEPI (06/2025 – Present).
2. **Performing data preprocessing, feature engineering, and optimization** to enhance model accuracy.
3. **Applying AI techniques** in practical scenarios.
4. **Relevant coursework** during your B.Sc. in Computer Engineering, including Machine Learning, Artificial Intelligence, and Data Structures & Algorithms.
5. **Certifications and training** in:
   - Deep Learning and System Design Diploma at CSkilled (10/2025 – Present).
   - Microsoft Machine Learning at DEPI (06/2025 – 01/2026).
   - Machine Learning Diploma at CSkilled (06/2025 – 10/2025).

Additionally, you have hands-on experience with various machine learning and deep learning techniques, including CNNs, RNNs, GNNs, GAT, VAE, LSTMs, Transformers, supervised learning, and unsupervised learning.


## Follow-up Question for Consistency Test

In [29]:
question_follow_up = "Can you list Ahmed's academic background and the fields he studied?"

response_follow_up = rag_chain.invoke(question_follow_up)

print(response_follow_up.content)

Ahmed's academic background includes:

- **Education**:  
  - **Faculty of Engineering, Zagazig University**  
    - **Degree**: B.Sc. in Computer Engineering  
    - **Years**: 2022 – 2027  
    - **Relevant Coursework**: Machine Learning, Artificial Intelligence, Data Structures & Algorithms, Database, Operating Systems, Computer Organization, Electronics, Embedded Systems, and Networks.  

The fields he studied primarily focus on **Computer Engineering**, with a strong emphasis on **Machine Learning**, **Artificial Intelligence**, and related areas such as **Data Structures**, **Algorithms**, and **Embedded Systems**.


# **Part 2 — Ask Questions About a Book**

In [30]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/RAG Chunking Techniques.pdf")

AI_IN_DRUGS = loader.load()

print("Number of pages:", len(AI_IN_DRUGS))

print(AI_IN_DRUGS[0].page_content[:500])

Number of pages: 8
AI Coach John Follow
  Repost
5
CHUNKING
TECHNIQUES
That save your 
RAG pipeline in production
Stop using only fixed chunking + overlap.
Production RAG needs smarter strategies.

Broken
Retrieval
→ 
Smart
Chunking
→ 
Clean
Pipeline


In [31]:
rag_chain2,retriever2=AI(AI_IN_DRUGS,100,10)

In [32]:
question = "What is the document about?"

response = rag_chain2.invoke(question)

print(response.content)

The document is about a process for splitting multi-topic documents into chunks based on semantic topic changes and hierarchical structure, while preserving the document's structure. It also mentions the impact of interest rates on investments as an example of a topic within a document.


In [33]:
question = "what are techniques of chinking"

response = rag_chain2.invoke(question)

print(response.content)

Based on the context provided, the techniques of chunking mentioned are:

1. **Fixed / Sliding Window Chunking**  
2. **Semantic Chunking**  
3. **Hierarchical Chunking**  
4. **LLM Based Chunking**  
5. **Agentic Chunking** (described as adaptive chunking)


In [34]:
query = "what is agentic chunking"

retrieved_docs = retriever2.invoke(query)

for i, doc in enumerate(retrieved_docs):

    print(f"\n--- Document {i+1} ---")

    print(doc.page_content)

    print("Metadata:", doc.metadata)


--- Document 1 ---
AI Coach John Follow
  Repost
 TECHNIQUE 5
Agentic Chunking
 WHAT IT IS
Adaptive chunking.
Metadata: {'creator': 'PyPDF', 'producer': 'PyPDF', 'total_pages': 8, 'creationdate': '', 'page': 5, 'page_label': '6', 'source': '/content/RAG Chunking Techniques.pdf'}

--- Document 2 ---
AI Coach John Follow
  Repost
 TECHNIQUE 2
Semantic Chunking
 WHAT IT IS
Metadata: {'page_label': '3', 'page': 2, 'creationdate': '', 'total_pages': 8, 'creator': 'PyPDF', 'producer': 'PyPDF', 'source': '/content/RAG Chunking Techniques.pdf'}

--- Document 3 ---
AI Coach John Follow
  Repost
 TECHNIQUE 3
Hierarchical Chunking
 WHAT IT IS
Metadata: {'page': 3, 'page_label': '4', 'total_pages': 8, 'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '/content/RAG Chunking Techniques.pdf'}

--- Document 4 ---
AI Coach John Follow
  Repost
 TECHNIQUE 4
LLM Based Chunking
 WHAT IT IS
Let an LLM decide:
Metadata: {'page_label': '5', 'total_pages': 8, 'creator': 'PyPD

In [35]:
rag_chain3,retriever3=AI(AI_IN_DRUGS,1200,100)

In [36]:
query = "How is deep learning used in drug discovery according to the document?"

retrieved_docs = retriever3.invoke(query)

for i, doc in enumerate(retrieved_docs):

    print(f"\n--- Document {i+1} ---")

    print(doc.page_content)

    print("Metadata:", doc.metadata)


--- Document 1 ---
Machine learning allows...
Deep learning uses neural
networks...
↓
OUTPUT
Chunk 1
Metadata: {'creator': 'PyPDF', 'total_pages': 8, 'source': '/content/RAG Chunking Techniques.pdf', 'creationdate': '', 'page': 1, 'page_label': '2', 'producer': 'PyPDF'}

--- Document 2 ---
Chunk 2
Machine learning allows...
Deep learning uses neural
networks...
(Overlap keeps context.)
Metadata: {'producer': 'PyPDF', 'page_label': '2', 'source': '/content/RAG Chunking Techniques.pdf', 'creationdate': '', 'creator': 'PyPDF', 'page': 1, 'total_pages': 8}

--- Document 3 ---
SKILLS
Languages
•C
•C++
•Java
•Python
Tools
•Git & Github
•Visual studio code
•Active VHDL
Software Libraries
•Numpy
•Pandas
•Matplotlib
•Seaborn
•Scikit_learn
•OpenCv
•Tensorflow
•Pytorch
Deep Learning & 
Machine Learning
•CNNS, RNNS, GNNS,
•GAT, VAE
•LSTMS
•Transformers 
•Supervised learning
•Unsupervised 
learning
Software Basics
•Problem solving
•OOP 
•DSA
•Databases 
CERTIFICATIONS & TRAINING
Deep Learning and 

## A Ques that doesnot exist

In [37]:
question = "Where does the sunrise"

response = rag_chain2.invoke(question)

print(response.content)

I don't know. The context provided does not contain any information about where the sunrise occurs.


# **Part 3 — Multi-PDF RAG**

In [38]:
from langchain_community.document_loaders import PyPDFLoader
import glob

documents = []

for pdf in glob.glob("/content/1780680628897.pdf"):
    loader = PyPDFLoader(pdf)
    documents.extend(loader.load())

print("Total pages:", len(documents))

Total pages: 10


In [39]:
from langchain_cohere import CohereEmbeddings

embeddings = CohereEmbeddings(
    model="embed-v4.0",
    cohere_api_key=key
)
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=30
)

chunks = text_splitter.split_documents(documents)

for doc in chunks[:5]:
    print(doc.metadata)

{'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-02-09T17:20:24+00:00', 'title': 'Evaluation Metrics in RAG', 'moddate': '2026-02-09T17:20:22+00:00', 'keywords': 'DAHA2EKN-K8,BAG0onMTyVQ,0', 'author': 'Statfusion AI', 'source': '/content/1780680628897.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1'}
{'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-02-09T17:20:24+00:00', 'title': 'Evaluation Metrics in RAG', 'moddate': '2026-02-09T17:20:22+00:00', 'keywords': 'DAHA2EKN-K8,BAG0onMTyVQ,0', 'author': 'Statfusion AI', 'source': '/content/1780680628897.pdf', 'total_pages': 10, 'page': 1, 'page_label': '2'}
{'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-02-09T17:20:24+00:00', 'title': 'Evaluation Metrics in RAG', 'moddate': '2026-02-09T17:20:22+00:00', 'keywords': 'DAHA2EKN-K8,BAG0onMTyVQ,0', 'author': 'Statfusion AI', 'source': '/content/1780680628897.pdf', 'total_pages': 10, 'page': 1, 'page_label': '2'}
{'producer': 'Canva', 'creator': '

In [40]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents= chunks[:100],
    embedding=embeddings,
    collection_name="pdf_rag"
)


In [41]:

retriever = vectorstore.as_retriever(
  search_kwargs={
      "k": 5
  }
)

from langchain_cohere import ChatCohere

llm = ChatCohere(
    model="command-a-03-2025",
    temperature=0,
    cohere_api_key=key
)


from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer the user's question using ONLY the context below.

If the answer cannot be found in the context,
say "I don't know."

Context:
{context}## top 1

Question:
{question}
""")

from langchain_core.runnables import RunnablePassthrough
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )


rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

In [42]:
# =========================
# PDF 1
# =========================

pdf1_source = "/content/1780680628897.pdf"

question = "how to evaluate RAG "

retriever_pdf1 = vectorstore.as_retriever(
    search_kwargs={
        "k": 5,
        "filter": {"source": pdf1_source}
    }
)

docs = retriever_pdf1.invoke(question)

response = rag_chain.invoke(question)

print("PDF 1:")
print(response.content)


# =========================
# PDF 2
# =========================

pdf2_source = "/content/Ahmed-Maged-Motea-FlowCV-Resume-20251008.pdf"

question = "what he is role working"

retriever_pdf2 = vectorstore.as_retriever(
    search_kwargs={
        "k": 5,
        "filter": {"source": pdf2_source}
    }
)

response = rag_chain.invoke(question)

print("\nPDF 2:")
print(response.content)


# =========================
# PDF 3
# =========================

pdf3_source = "/content/RAG Chunking Techniques.pdf"

question = "give me 2 types of chunkking"

retriever_pdf3 = vectorstore.as_retriever(
    search_kwargs={
        "k": 5,
        "filter": {"source": pdf3_source}
    }
)

response = rag_chain.invoke(question)

print("\nPDF 3:")
print(response.content)




# =========================
# PDF 1 + PDF 2
# =========================

question = "what is relation betwen chunking and evaluating rag"

response = rag_chain.invoke(question)

print("\nPDF 1 + PDF 2:")
print(response.content)


# =========================
# ALL PDFs
# =========================

question = "is ahmed maged rag enginner"

response = rag_chain.invoke(question)

print("\nALL PDFs:")
print(response.content)


# =========================
# NOT IN ANY PDF
# =========================

question = "What clinical trial results for COVID-19 vaccines are reported in these papers?"

response = rag_chain.invoke(question)

print("\nNOT IN THE DOCUMENTS:")
print(response.content)

PDF 1:
RAG evaluation happens at two levels:

1. **Retriever Evaluation**: Measures how good the retriever is at finding useful chunks. This includes metrics like:
   - **Recall@K**: Measures if the correct information is among the top K retrieved chunks.
   - **Precision@K**: Measures the proportion of relevant chunks among the top K retrieved chunks.
   - **Context Relevancy**: Assesses how relevant the retrieved chunks are to the query.

2. **Generator (LLM) Evaluation**: Measures how good the LLM’s answer is, focusing on:
   - **Faithfulness**: Ensures the answer is faithful to the retrieved context.
   - **Answer Relevancy**: Checks if the answer is relevant to the query.
   - **Groundedness**: Ensures the answer is grounded in the retrieved context and does not hallucinate.

**Tools for RAG Evaluation**:
- **Ragas**: Provides specialized metrics like faithfulness and context relevance.
- **TruLens**: Tracks grounding and allows for feedback.
- **DeepEval**: Offers LLM-based scori

# 🎯 Part 4: Evaluating the RAG Retriever

Before testing the chatbot's final answers, we must ensure our **retrieval engine** is actually finding the right information. We evaluate this independently from the LLM using classic Information Retrieval (IR) metrics:

### 📊 Key Metrics
* **Precision@k:** *How much of what we retrieved is useful?*
  (Percentage of the top-k retrieved chunks that are actually relevant).
* **Recall@k:** *Did we miss anything important?*
  (Percentage of all known relevant sources that successfully made it into the top-k).
* **Average Precision (AP):** *Is the best information at the top?*
  (Rewards the system for ranking relevant chunks higher in the search results).
* **Mean Average Precision (MAP):**
  (The average AP across all test questions, providing a single summary score for the retriever's overall quality).

---

### 🧪 Evaluation Methodology

To calculate these scores, we build a **"Golden" Test Set**:
1. We define a list of test questions.
2. We pair each question with the specific PDF(s) that we *know* contain the answer (e.g., `pdf1_source`, `pdf2_source`).

> **📌 Relevance Rule:** A retrieved chunk is only counted as "relevant" if its source metadata matches the designated correct PDF for that specific question.

## Step 1 — Build the golden (ground-truth) test set

In [48]:
# Reuse the same source paths defined in Part 3
golden_set = [
    {
        "question": "What are the main applications of artificial intelligence in drug discovery discussed in this paper?",
        "relevant_sources": [pdf1_source]
    },
    {
        "question": "What deep learning and graph learning methods are discussed for drug-drug interaction prediction?",
        "relevant_sources": [pdf2_source]
    },
    {
        "question": "How are generative AI models used for de novo drug design according to this paper?",
        "relevant_sources": [pdf3_source]
    },
    {
        "question": "How do the AI approaches discussed in the first paper differ from the deep and graph learning approaches discussed in the second paper?",
        "relevant_sources": [pdf1_source, pdf2_source]
    },
    {
        "question": "Compare the AI approaches discussed across all three papers and explain how they can contribute to different stages of drug discovery.",
        "relevant_sources": [pdf1_source, pdf2_source, pdf3_source]
    },
    {
        "question": "What clinical trial results for COVID-19 vaccines are reported in these papers?",
        "relevant_sources": []  # not present in any document -> nothing should count as relevant
    },
]
len(golden_set)

6

## Step 2 — Precision@k, Recall@k and Average Precision functions

In [44]:
def get_retrieved_sources(retriever, question, k):
    """Retrieve top-k chunks for a question and return the ordered list of their source PDFs."""
    docs = retriever.invoke(question)[:k]
    return [d.metadata.get("source") for d in docs]

def precision_at_k(retrieved_sources, relevant_sources, k):
    if k == 0:
        return 0.0
    top_k = retrieved_sources[:k]
    hits = sum(1 for s in top_k if s in relevant_sources)
    return hits / k

def recall_at_k(retrieved_sources, relevant_sources, k):
    if len(relevant_sources) == 0:
        # Nothing relevant exists -> recall is only meaningful if we also retrieved nothing relevant
        return 1.0 if precision_at_k(retrieved_sources, relevant_sources, k) == 0 else 0.0
    top_k = retrieved_sources[:k]
    unique_relevant_hit = {s for s in top_k if s in relevant_sources}
    return len(unique_relevant_hit) / len(set(relevant_sources))

def average_precision(retrieved_sources, relevant_sources):
    """Standard AP: average of precision@i computed at every rank i where a relevant item appears."""
    if len(relevant_sources) == 0:
        # No relevant docs exist: AP = 1 if we retrieved none of them (correctly), else 0
        return 1.0 if not any(s in relevant_sources for s in retrieved_sources) else 0.0
    hits = 0
    precisions = []
    for i, s in enumerate(retrieved_sources, start=1):
        if s in relevant_sources:
            hits += 1
            precisions.append(hits / i)
    if not precisions:
        return 0.0
    return sum(precisions) / len(relevant_sources)

## Step 3 — Run the evaluation over the golden set

In [50]:
def evaluate_retriever(retriever, golden_set, k=5):
    rows = []
    for item in golden_set:
        question = item["question"]
        relevant_sources = item["relevant_sources"]
        retrieved_sources = get_retrieved_sources(retriever, question, k)
        p_at_k = precision_at_k(retrieved_sources, relevant_sources, k)
        r_at_k = recall_at_k(retrieved_sources, relevant_sources, k)
        ap = average_precision(retrieved_sources, relevant_sources)
        rows.append({
            "question": question,
            "precision@k": round(p_at_k, 3),
            "recall@k": round(r_at_k, 3),
            "AP": round(ap, 3)
        })
    mean_precision = sum(r["precision@k"] for r in rows) / len(rows)
    mean_recall = sum(r["recall@k"] for r in rows) / len(rows)
    mean_ap = sum(r["AP"] for r in rows) / len(rows)  # this is MAP
    return rows, mean_precision, mean_recall, mean_ap

results, mean_precision, mean_recall, MAP = evaluate_retriever(retriever, golden_set, k=5)
for r in results:
    print(f"Q: {r['question']}")
    print(f"   Precision@5 = {r['precision@k']} | Recall@5 = {r['recall@k']} | AP = {r['AP']}")
    print()
print("=" * 50)
print(f"Mean Precision@5 : {mean_precision:.3f}")
print(f"Mean Recall@5    : {mean_recall:.3f}")
print(f"MAP              : {MAP:.3f}")

Q: What are the main applications of artificial intelligence in drug discovery discussed in this paper?
   Precision@5 = 0.0 | Recall@5 = 0.0 | AP = 0.0

Q: What deep learning and graph learning methods are discussed for drug-drug interaction prediction?
   Precision@5 = 0.4 | Recall@5 = 1.0 | AP = 0.833

Q: How are generative AI models used for de novo drug design according to this paper?
   Precision@5 = 0.8 | Recall@5 = 1.0 | AP = 3.55

Q: How do the AI approaches discussed in the first paper differ from the deep and graph learning approaches discussed in the second paper?
   Precision@5 = 0.0 | Recall@5 = 0.0 | AP = 0.0

Q: Compare the AI approaches discussed across all three papers and explain how they can contribute to different stages of drug discovery.
   Precision@5 = 1.0 | Recall@5 = 0.333 | AP = 1.667

Q: What clinical trial results for COVID-19 vaccines are reported in these papers?
   Precision@5 = 0.0 | Recall@5 = 1.0 | AP = 1.0

Mean Precision@5 : 0.367
Mean Recall@5    

## Step 4 — (Optional) Compare k values and chunk sizes

In [51]:
# You can reuse this to see how MAP changes with different k, or with different# chunk_size/chunk_overlap combinations (rebuild the vectorstore with the AI() function# from earlier, then pass the new retriever in here).for k_value in [1, 3, 5, 10]:    _, p, r, m = evaluate_retriever(retriever, golden_set, k=k_value)    print(f"k={k_value:>2} -> Precision={p:.3f} | Recall={r:.3f} | MAP={m:.3f}")

**Interpretation notes:**- A **high Precision, low Recall** retriever returns mostly-correct chunks but misses some relevant PDFs (often happens with a small `k`).- A **high Recall, low Precision** retriever grabs most of the relevant material but also drags in noise (often happens with a large `k` or a small `chunk_size` producing many near-duplicate chunks).- **MAP** is the best single number to compare configurations (different `k`, `chunk_size`, `chunk_overlap`) because it accounts for *ranking order*, not just whether the relevant chunk was retrieved anywhere in the list.